In [1]:
import pandas as pd
import sqlite3
import requests
import os

import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

In [2]:
def fetch(query, conn, formatted=True):
    # execute the query and fetch all rows
    cur = conn.cursor()
    cur.execute(query)
    rs = cur.fetchall()
    
    # extract column names from the cursor description
    columns = [desc[0] for desc in cur.description]
    
    # return a dataframe with column names
    return pd.DataFrame(rs, columns=columns) if formatted else rs

def show_tables(conn):
    return [x[0] for x in fetch('SELECT tbl_name FROM sqlite_master WHERE type="table"', conn, False)]

def shape(table, conn):
    nrows = fetch(f'SELECT COUNT(*) FROM {table}', conn, False)[0][0]
    ncols = fetch(f'SELECT COUNT(*) FROM pragma_table_info("{table}")', conn, False)[0][0]

    return (nrows, ncols)

def desc(table, conn):
    cur = conn.cursor()
    cur.execute(f'PRAGMA table_info("{table}")')
    columns = [row[1] for row in cur.fetchall()]
    
    return columns

def info(table, conn):
    # table constraints (domain, null, default, pk)
    df1 = fetch(f'PRAGMA table_info("{table}")', conn)
    columns = desc(table, conn)
    
    # entries per column
    counts = ', '.join([f'COUNT(*) AS "{column}"' for column in columns])
    df2 = fetch(f'SELECT {counts} FROM "{table}"', conn).transpose()
    df2.columns = ['count']
    
    # non-null entries per column
    counts = ', '.join([f'COUNT("{column}") AS "{column}"' for column in columns])
    df3 = fetch(f'SELECT {counts} FROM "{table}"', conn).transpose()
    df3.columns = ['notnull count']

    # unique non-null entries per column
    counts = ', '.join([f'COUNT(DISTINCT "{column}") AS "{column}"' for column in columns])
    df4 = fetch(f'SELECT {counts} FROM "{table}"', conn).transpose()
    df4.columns = ['unique count']
    
    return df1.merge(df2, left_on='name', right_index=True) \
            .merge(df3, left_on='name', right_index=True) \
            .merge(df4, left_on='name', right_index=True)

def display(urls, cols=5):
    # fetch images
    images = []
    for url in urls:
        response = requests.get(url)
        if response.status_code == 200:
            images.append(Image.open(BytesIO(response.content)))

    # calculate the number of rows
    rows = (len(images) + cols - 1) // cols  
    fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))

    for i, ax in enumerate(axs.flat):
        if i < len(images):
            ax.imshow(images[i])
            ax.axis("off")  # Hide axes
        else:
            ax.axis("off")  # Hide unused subplots
    plt.tight_layout()
    plt.show()

def hist(data, xlabel='', ylabel='', bins='auto'):
    # convert list of tuples into dataframe
    df = pd.DataFrame(data, columns=['key', 'frequency'])
    # expand the data based on frequencies
    expanded = df.loc[df.index.repeat(df['frequency'])].reset_index(drop=True)

    # plot histogram
    sns.histplot(expanded['key'], bins=bins, kde=False)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=90)
    plt.show()

In [3]:
db_name = 'importacoes_brasil_2024.db'

# Remover banco existente se houver
if os.path.exists(db_name):
    os.remove(db_name)
    print(f"🗑️  Banco existente removido: {db_name}")


🗑️  Banco existente removido: importacoes_brasil_2024.db


In [4]:

conn = sqlite3.connect(db_name)
cur = conn.cursor()

In [5]:
DDL = [
    """ 
        CREATE TABLE IF NOT EXISTS Unidade (
            COD_UNID INTEGER PRIMARY KEY,
            NOME_UNID TEXT NOT NULL,
            SIGLA_UNID TEXT NOT NULL
        );
    """,

    """
        CREATE TABLE IF NOT EXISTS Mes (
            COD_MES INTEGER PRIMARY KEY AUTOINCREMENT,
            TRIMESTRE INTEGER,
            SEMESTRE INTEGER,
            NOME_MES NOT NULL
        );        
    """,

    """  
        CREATE TABLE IF NOT EXISTS UF (
            COD_UF INTEGER PRIMARY KEY AUTOINCREMENT,
            SIGLA_UF TEXT NOT NULL,
            NOME_UF TEXT NOT NULL
        );
    """,

    """
        CREATE TABLE IF NOT EXISTS NCM (
            COD_NCM TEXT PRIMARY KEY,
            COD_UNID INTEGER NOT NULL,
            NOME_NCM TEXT NOT NULL,
            FOREIGN KEY (COD_UNID) REFERENCES Unidade(COD_UNID)
        );
    """,

    """ 
        CREATE TABLE IF NOT EXISTS Pais (
            COD_PAIS INTEGER PRIMARY KEY,
            SIGLA_PAIS TEXT,
            NOME_PAIS TEXT NOT NULL
        );
    """, 

    """ 
        CREATE TABLE IF NOT EXISTS URF (
            COD_URF TEXT PRIMARY KEY,
            NOME_URF TEXT NOT NULL
        );
    """,

    """ 
        CREATE TABLE IF NOT EXISTS Via (
            COD_VIA INTEGER PRIMARY KEY,
            NOME_VIA TEXT NOT NULL
        );
    """,

    """  
        CREATE TABLE IF NOT EXISTS Importacoes (
            ID_IMPORTACAO INTEGER PRIMARY KEY AUTOINCREMENT,
            COD_MES INTEGER NOT NULL,
            COD_NCM TEXT NOT NULL,
            COD_UNID INTEGER NOT NULL,
            COD_PAIS INTEGER NOT NULL,
            COD_UF INTEGER NOT NULL,
            COD_VIA INTEGER NOT NULL,
            COD_URF TEXT NOT NULL,
            QT_ESTATISTICA INTEGER NOT NULL,
            KG_LIQUIDO INTEGER NOT NULL,
            VL_FOB INTEGER NOT NULL,
            VL_FRETE INTEGER NOT NULL,
            VL_SEGURO INTEGER NOT NULL,

            FOREIGN KEY (COD_MES) REFERENCES Mes(COD_MES),
            FOREIGN KEY (COD_NCM) REFERENCES NCM(COD_NCM),
            FOREIGN KEY (COD_UNID) REFERENCES Unidade(COD_UNID),
            FOREIGN KEY (COD_PAIS) REFERENCES Pais(COD_PAIS),
            FOREIGN KEY (COD_UF) REFERENCES UF(COD_UF),
            FOREIGN KEY (COD_VIA) REFERENCES Via(COD_VIA),
            FOREIGN KEY (COD_URF) REFERENCES URF(COD_URF)
        );
    """
]

with sqlite3.connect(db_name) as conn:
    cur = conn.cursor()
    cur.execute("PRAGMA foreing_keys= ON")

    for ddl in DDL:
        cur.execute(ddl)
    
    conn.commit()


In [6]:
show_tables(conn)


['Unidade',
 'Mes',
 'sqlite_sequence',
 'UF',
 'NCM',
 'Pais',
 'URF',
 'Via',
 'Importacoes']

In [7]:
# Populando tabela Mes
cur.execute("""  
    INSERT INTO Mes (COD_MES, TRIMESTRE, SEMESTRE, NOME_MES) VALUES
        (1, 1, 1, 'Janeiro'),
        (2, 1, 1, 'Fevereiro'),
        (3, 1, 1, 'Março'),
        (4, 2, 1, 'Abril'),
        (5, 2, 1, 'Maio'),
        (6, 2, 1, 'Junho'),
        (7, 3, 2, 'Julho'),
        (8, 3, 2, 'Agosto'),
        (9, 3, 2, 'Setembro'),
        (10, 4, 2, 'Outubro'),
        (11, 4, 2, 'Novembro'),
        (12, 4, 2, 'Dezembro');  
""")


In [8]:
# Populando tabela UF
cur.execute(""" 
INSERT INTO UF (COD_UF, SIGLA_UF, NOME_UF) VALUES
        (1, 'AC', 'Acre'),
        (2, 'AL', 'Alagoas'),
        (3, 'AM', 'Amazonas'),
        (4, 'AP', 'Amapá'),
        (5, 'BA', 'Bahia'),
        (6, 'CE', 'Ceará'),
        (7, 'DF', 'Distrito Federal'),
        (8, 'ES', 'Espírito Santo'),
        (9, 'GO', 'Goiás'),
        (10, 'MA', 'Maranhão'),
        (11, 'MG', 'Minas Gerais'),
        (12, 'MS', 'Mato Grosso do Sul'),
        (13, 'MT', 'Mato Grosso'),
        (14, 'PA', 'Pará'),
        (15, 'PB', 'Paraíba'),
        (16, 'PE', 'Pernambuco'),
        (17, 'PI', 'Piauí'),
        (18, 'PR', 'Paraná'),
        (19, 'RJ', 'Rio de Janeiro'),
        (20, 'RN', 'Rio Grande do Norte'),
        (21, 'RO', 'Rondônia'),
        (22, 'RR', 'Roraima'),
        (23, 'RS', 'Rio Grande do Sul'),
        (24, 'SC', 'Santa Catarina'),
        (25, 'SE', 'Sergipe'),
        (26, 'SP', 'São Paulo'),
        (27, 'TO', 'Tocantins');
 """)


In [9]:
def migrate_ncm_unid(path, conn):
    # Separando NCM e UNID de ncm_unid.csv

    cur = conn.cursor()

    df = pd.read_csv(path, dtype={'CO_NCM': str})

    df['NO_NCM_POR'] = df['NO_NCM_POR'].str.strip().replace({'': None})
    df['NO_UNID']    = df['NO_UNID'].str.strip().replace({'': None})
    df['SG_UNID']    = df['SG_UNID'].str.strip().replace({'': None})

    unidade_rows = (
        df[['CO_UNID', 'NO_UNID', 'SG_UNID']]
        .drop_duplicates()
        .dropna()
        .astype({'CO_UNID': int})
        .values.tolist()
    )

    cur.executemany(""" 
        INSERT INTO Unidade (COD_UNID, NOME_UNID, SIGLA_UNID)
        VALUES(?, ?, ?)
    """, unidade_rows)

    ncm_rows = (
        df[['CO_NCM', 'CO_UNID', 'NO_NCM_POR']]
        .drop_duplicates()
        .dropna(subset=['CO_NCM', 'CO_UNID'])  
        .astype({'CO_UNID': int})
        .values.tolist()
    )

    cur.executemany("""
        INSERT INTO NCM (COD_NCM, COD_UNID, NOME_NCM)
        VALUES (?, ?, ?)
    """, ncm_rows)

    conn.commit()
    print(f">>> Migrando {len(unidade_rows)} unidades e {len(ncm_rows)} NCM entries.")


    

In [10]:
migrate_ncm_unid('data/ncm_unid.csv', conn)

>>> Migrando 13 unidades e 13721 NCM entries.


In [11]:
def migrate_pais(path, conn):
    df = pd.read_csv(path)

    df = df[['CO_PAIS', 'CO_PAIS_ISOA3', 'NO_PAIS']].copy()
    df.columns = ['COD_PAIS', 'SIGLA_PAIS', 'NOME_PAIS']

    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.replace({'': None})

    cur = conn.cursor()
    data = df.values.tolist()

    cur.executemany("""
        INSERT INTO Pais (COD_PAIS, SIGLA_PAIS, NOME_PAIS)
        VALUES (?, ?, ?)
    """, data)
    conn.commit()
    print(f">>> Migrando {len(data)} linhas em 'Pais'")

migrate_pais('data/pais.csv', conn)

>>> Migrando 281 linhas em 'Pais'


C:\Users\pedro\AppData\Local\Temp\ipykernel_10276\1611907022.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [12]:
def migrate_via(path, conn):
    df = pd.read_csv(path)

    df = df[['CO_VIA', 'NO_VIA']].copy()
    df.columns = ['COD_VIA', 'NOME_VIA']

    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.replace({'': None})

    cur = conn.cursor()
    data = df.values.tolist()

    cur.executemany("""
        INSERT INTO Via (COD_VIA, NOME_VIA)
        VALUES (?, ?)
    """, data)
    conn.commit()
    print(f">>> Migrando {len(data)} linhas em 'Via'")

migrate_via('data/via.csv', conn)

>>> Migrando 17 linhas em 'Via'


C:\Users\pedro\AppData\Local\Temp\ipykernel_10276\3175653566.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [13]:
def migrate_urf(path, conn):
    df = pd.read_csv(path, dtype={'CO_URF': str})

    df = df[['CO_URF', 'NO_URF']].copy()
    df.columns = ['COD_URF', 'NOME_URF']

    
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.replace({'': None})

    df['NOME_URF'] = df['NOME_URF'].str.split(' - ', n=1).str[1]
    
    cur = conn.cursor()
    data = df.values.tolist()

    cur.executemany(""" 
        INSERT INTO URF (COD_URF, NOME_URF)
        VALUES (?, ?)
    """, data)

    conn.commit()

    print(f">>> Migrando {len(data)} linhas em 'URF'")

migrate_urf('data/urf.csv', conn)


>>> Migrando 279 linhas em 'URF'


C:\Users\pedro\AppData\Local\Temp\ipykernel_10276\2097534099.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [14]:
# list = list(show_tables(conn))
desc('Importacoes', conn)

['ID_IMPORTACAO',
 'COD_MES',
 'COD_NCM',
 'COD_UNID',
 'COD_PAIS',
 'COD_UF',
 'COD_VIA',
 'COD_URF',
 'QT_ESTATISTICA',
 'KG_LIQUIDO',
 'VL_FOB',
 'VL_FRETE',
 'VL_SEGURO']

In [15]:
def fetch_uf_map(conn):
    cur = conn.cursor()
    cur.execute("SELECT SIGLA_UF, COD_UF FROM UF")
    return dict(cur.fetchall())


In [16]:
def migrate_importacoes(path, conn):
    uf_map = fetch_uf_map(conn)
    cur = conn.cursor()

    cur.execute("PRAGMA foreign_keys= ON")

    insert_query = """ 
        INSERT INTO Importacoes (
            COD_MES, COD_NCM, COD_UNID, COD_PAIS, COD_UF,
            COD_VIA, COD_URF, QT_ESTATISTICA, KG_LIQUIDO,
            VL_FOB, VL_FRETE, VL_SEGURO
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) """

    total_inserted = 0
    total_skipped = 0 # Faltando SG_UF_NCM

    for batch in pd.read_csv(path, sep= ';', dtype={'CO_NCM': str, 'CO_URF': str}, chunksize= 5000):
        batch['SG_UF_NCM'] = batch['SG_UF_NCM'].str.strip()
        batch['COD_UF'] = batch['SG_UF_NCM'].map(uf_map)

        invalid_rows = batch['COD_UF'].isna().sum()
        total_skipped += invalid_rows
        valid = batch.dropna(subset=['COD_UF'])

        rows = [
            (
                row.CO_MES,
                row.CO_NCM,
                row.CO_UNID,
                row.CO_PAIS,
                int(row.COD_UF),
                row.CO_VIA,
                row.CO_URF,
                row.QT_ESTAT,
                row.KG_LIQUIDO,
                row.VL_FOB,
                row.VL_FRETE,
                row.VL_SEGURO
            )
            for row in valid.itertuples(index=False)
        ]

        if rows:
            cur.execute("BEGIN")
            cur.executemany(insert_query, rows)
            conn.commit()
            total_inserted += len(rows)

    print(f">>> Inserted: {total_inserted} rows")
    print(f">>> Skipped: {total_skipped} rows (missing UF code)")

In [17]:
migrate_importacoes('data/IMP_2024.csv', conn)



>>> Inserted: 2273687 rows
>>> Skipped: 21 rows (missing UF code)


In [18]:
cur.execute(""" 
    UPDATE URF
    SET NOME_URF = TRIM(CASE
            WHEN INSTR(NOME_URF, ' - ') > 0 THEN SUBSTR(NOME_URF, INSTR(NOME_URF, ' - ') + 3)
            ELSE NOME_URF
        END);
 """)

conn.commit()


In [19]:
df = fetch('select * from URF', conn, True)
df
cur.close()
conn.close()
